# Extending state anchors via a two-sided random walk

`construct_states_prior` discloses the latent state at a handful of anchor timesteps and leaves every
other step at `P_obs = inf` (undisclosed). For a single channel touched by no other data, those null
steps are **not** truly uninformed: the driftless random-walk transition already implies an exact
marginal that propagates each anchor forward *and* backward,

$$a_{\text{obs}}[t] = a^{*}, \qquad P_{\text{obs}}[t] = P^{*} + |t - t^{*}|\,Q,$$

where $(a^{*}, P^{*})$ is the nearest disclosed anchor (minimum variogram distance $|t - t^{*}|$) and
$Q = \sigma_q^2$ is the per-state process variance. The mean is carried flat; the variance grows
linearly with the lag, forming a "cone" that pinches to $P^{*}$ at the anchor.

`extend_states_prior_nearest(ssp_priors, Q)` fills the `inf` entries with this marginal. It is the exact
marginal only for an **isolated** channel, so treat the output as a prior fed into the
augmented-measurement step — not a final posterior. States with no anchor stay `inf`, and $Q \to \infty$
recovers the un-extended prior (only anchors informed).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from jax import numpy as jnp

from bunobee.models.ssp.plotting import plot_prior_heatmap, plot_states
from bunobee.models.ssp.prior import disclosed_idx, extend_states_prior_nearest
from bunobee.simulation.ssp import construct_states_prior


## 1. Construct the disclosed states prior

We synthesise a weekly (52-step) prior over an level plus three media regressors, disclosing the
ground-truth latent state over three random windows. The level carries no finite-variance anchor
(its variance stays `inf`), which later demonstrates the "no anchor -> stays undisclosed" behaviour.

In [ ]:
regressors = ["tv", "search", "social"]
state_labels = ["level", *regressors]
n_states = len(state_labels)
n_steps = 52

true_states = jnp.array([0.0, 0.8, 0.5, 0.3])  # level ignored (var stays inf)

states_prior = construct_states_prior(
    n_steps=n_steps,
    n_states=n_states,
    true_states=true_states,
    regressors=regressors,
    n_periods=3,
    n_points=4,
    seed=7,
    obs_scale=0.1,
)

dates = pd.date_range("2024-01-01", periods=n_steps, freq="W")
print("disclosed anchor steps:", disclosed_idx(states_prior))
print("finite fraction (base):", np.isfinite(states_prior["P_obs"].values).mean().round(3))

In [ ]:
plot_prior_heatmap(states_prior, quantity="both")
plt.show()

## 2. Extend the prior along the random walk

`extend_states_prior_nearest` fills every undisclosed step of each anchored state with the nearest-anchor
random-walk marginal. Here we use a single scalar process variance `Q`; a length-`n_states` array can
instead give each state its own slope.

In [ ]:
# Extend the disclosed states prior and visualise the result
Q = 0.02  # per-state process variance sigma_q**2 (scalar broadcast to every state)
ext_states_prior = extend_states_prior_nearest(states_prior, Q)

print("finite fraction (extended):", np.isfinite(ext_states_prior["P_obs"].values).mean().round(3))
print("level stays fully undisclosed:", bool(np.all(np.isinf(ext_states_prior["P_obs"].values[:, 0]))))

# Plot the extended time-point prior as a states x time heatmap
plot_prior_heatmap(ext_states_prior, quantity="both")
plt.show()

In [ ]:
# Sanity checks moved here for clarity (can be removed if not needed)
# Sanity check: an undisclosed step matches P* + |t - t*|·Q for its nearest anchor.
tv = state_labels.index("tv")
anchors = disclosed_idx(states_prior)  # same disclosure windows for every regressor
for t in (10, 20, 40):
    t_star = int(anchors[np.argmin(np.abs(anchors - t))])
    expected = states_prior["P_obs"].values[t_star, tv] + abs(t - t_star) * Q
    got = ext_states_prior["P_obs"].values[t, tv]
    print(f"tv step t={t:>2}: nearest anchor t*={t_star}, P_obs={got:.3f} (expected {expected:.3f})")

ext_states_prior

## 3. Contrast the prior with vs. without the extension

We draw pseudo-samples from each prior's per-step marginal $\mathcal{N}(a_{\text{obs}}, P_{\text{obs}})$
and hand both to `plot_states`, overlaying them in a single figure. Undisclosed steps (`inf` variance)
become `NaN`, so the **anchors-only** ribbon shows only the disclosure windows, while the
**RW-extension** ribbon is a continuous cone that narrows to each anchor and widens with lag. Red
markers flag the disclosed anchors; the level panel stays empty (no anchor to extend).

In [ ]:
def prior_samples(ssp_priors, n_draws=2000, seed=0):
    """Draw samples from a prior's per-step Gaussian marginal.

    Parameters
    ----------
    ssp_priors : xr.Dataset
        Prior with ``a_obs`` / ``P_obs`` over ``(time, state)``.
    n_draws : int, optional
        Number of Monte Carlo draws, by default 2000.
    seed : int, optional
        RNG seed, by default 0.

    Returns
    -------
    np.ndarray, shape (n_draws, n_steps, n_states)
        Draws with undisclosed (``inf``-variance) steps set to ``NaN`` so the
        ribbon shows a gap there.
    """
    rng = np.random.default_rng(seed)
    a = np.asarray(ssp_priors["a_obs"].values, dtype=float)
    p = np.asarray(ssp_priors["P_obs"].values, dtype=float)
    draws = a[None] + np.sqrt(p)[None] * rng.standard_normal((n_draws, *a.shape))
    draws[:, ~np.isfinite(p)] = np.nan
    return draws


posterior = {
    "anchors only": prior_samples(states_prior),
    "RW extension": prior_samples(ext_states_prior),
}

fig, axes = plot_states(
    posterior,
    dates.values,
    state_labels,
    states_key=["anchors only", "RW extension"],
    obs_idx=disclosed_idx(states_prior),
    a_obs=states_prior["a_obs"].values,
    P_obs=states_prior["P_obs"].values,
    title=f"Two-sided random-walk extension of state anchors (Q={Q})",
    n_cols=2,
    colors={"anchors only": "darkgreen", "RW extension": "steelblue"},
)
plt.show()

## Takeaways

- The **RW extension** turns sparse anchors into a full time-point prior: the mean is held flat and
  the variance opens into a symmetric cone $P^{*} + |t - t^{*}|\,Q$ around each anchor.
- With multiple anchors per state, each step uses the **nearest** anchor; a state with **no** anchor
  (here the level) keeps `P_obs = inf`.
- Increasing `Q` widens the cone faster; $Q \to \infty$ collapses back to the anchors-only prior.
  Feed the extended prior into the augmented-measurement step for a quick de-biased read on an
  isolated channel.